## Task 1: Dataset Selection



In [13]:
pip install datasets==2.14.0

INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/566.4 kB ? eta -:--:--
   ---------------------------------------- 566.4/566.4 kB 5.3 MB/s eta 0:00:00

  Attempting uninstall: dill

    Found existing installation: dill 0.4.1

    Uninstalling dill-0.4.1:

      Successfully uninstalled dill-0.4.1

   ---------------------------------------- 0/4 [dill]
   ---------------------------------------- 0/4 [dill]
   ---------------------------------------- 0/4 [dill]
   ---------------------------------------- 0/4 [dill]
   ---------------------------------------- 0/4 [dill]
   ---------------------------------------- 0/4 [dill]
   ---------------------------------------- 0/4 [dill]
   ---------------------------------------- 0/4 [dill]
   ---------------------------------------- 0/4 [dill]
  Attempting uninstall: multiprocess
   ----------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.4.0 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
# Install required libraries
!pip install transformers datasets seqeval evaluate accelerate -q

## Import Libraries

In [3]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import evaluate
import warnings
warnings.filterwarnings('ignore')

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using device:", device)

PyTorch version: 2.11.0+cpu
CUDA available: False
Using device: cpu


## Task 2: Data Preprocessing

### 2.1 Load the CoNLL-2003 Dataset

In [14]:
from datasets import load_dataset
dataset = load_dataset("conll2003", trust_remote_code=True)
print(dataset)
print("\nFeatures:", dataset['train'].features)
print("\nSample:", dataset['train'][0])

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'conll2003' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


RuntimeError: Dataset scripts are no longer supported, but found conll2003.py

In [ ]:

pos_feature  = dataset['train'].features['pos_tags']
chunk_feature = dataset['train'].features['chunk_tags']

pos_label_list   = pos_feature.feature.names
chunk_label_list = chunk_feature.feature.names

print("POS Tags:", pos_label_list)
print(f"\nTotal POS labels: {len(pos_label_list)}")
print("\nChunk Tags:", chunk_label_list)
print(f"\nTotal Chunk labels: {len(chunk_label_list)}")

### 2.2 Initialize Tokenizer

In [ ]:
MODEL_CHECKPOINT = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
print("Tokenizer loaded:", MODEL_CHECKPOINT)

### 2.3 Label Alignment – Handling Subwords and Special Tokens



In [ ]:
def tokenize_and_align_labels(examples, label_column):
    """
    Tokenizes input words using the BERT tokenizer and aligns
    the word-level labels to the subword-level tokens.

    Special tokens ([CLS], [SEP]) get label = -100 (ignored in loss).
    Continuation subwords (##) also get label = -100.
    Only the FIRST subword of each word carries the real label.
    """
    # Tokenize: is_split_into_words=True because input is already word-tokenized
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    all_labels = []
    for i, label_ids in enumerate(examples[label_column]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_id = None
        aligned_labels = []

        for word_id in word_ids:
            if word_id is None:
                # Special tokens: [CLS], [SEP] → ignore in loss
                aligned_labels.append(-100)
            elif word_id != previous_word_id:
                # First subword of a new word → assign real label
                aligned_labels.append(label_ids[word_id])
            else:
                # Continuation subword (e.g., ##ing) → ignore in loss
                aligned_labels.append(-100)

            previous_word_id = word_id

        all_labels.append(aligned_labels)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs

print("Label alignment function defined.")

### 2.4 Visualize Label Alignment on a Sample Sentence

In [ ]:
# Show alignment for one example
sample = dataset['train'][0]
tokens = sample['tokens']
pos_tags = sample['pos_tags']

print("Original words:", tokens)
print("Original POS IDs:", pos_tags)
print("Original POS Tags:", [pos_label_list[t] for t in pos_tags])

# Tokenize just this sentence
tok_output = tokenizer(tokens, is_split_into_words=True)
subwords   = tokenizer.convert_ids_to_tokens(tok_output['input_ids'])
word_ids   = tok_output.word_ids()

print("\n{:<20} {:<12} {}".format("Subword Token", "Word ID", "Aligned POS"))
print("-" * 45)
prev = None
for sw, wid in zip(subwords, word_ids):
    if wid is None:
        label_str = "-100 (special)"
    elif wid != prev:
        label_str = pos_label_list[pos_tags[wid]]
    else:
        label_str = "-100 (subword)"
    print(f"{sw:<20} {str(wid):<12} {label_str}")
    prev = wid

### 2.5 Preprocess Full Dataset



In [ ]:
from functools import partial

# POS Tagging Dataset
tokenized_pos = dataset.map(
    partial(tokenize_and_align_labels, label_column="pos_tags"),
    batched=True,
    remove_columns=dataset["train"].column_names
)

# Chunking Dataset 
tokenized_chunk = dataset.map(
    partial(tokenize_and_align_labels, label_column="chunk_tags"),
    batched=True,
    remove_columns=dataset["train"].column_names
)

print("POS Dataset:", tokenized_pos)
print("\nChunk Dataset:", tokenized_chunk)

# Verify structure
sample_tok = tokenized_pos['train'][0]
print("\nTokenized sample keys:", list(sample_tok.keys()))
print("input_ids length:", len(sample_tok['input_ids']))
print("attention_mask:", sample_tok['attention_mask'][:10])
print("labels:", sample_tok['labels'][:10])

## Task 3: Model Setup


### 3.1 POS Tagging Model

In [ ]:
# Build label mappings for POS Tagging
pos_id2label = {i: label for i, label in enumerate(pos_label_list)}
pos_label2id = {label: i for i, label in enumerate(pos_label_list)}

print(f"Number of POS labels: {len(pos_label_list)}")
print("POS id2label (first 5):", dict(list(pos_id2label.items())[:5]))

# Load DistilBERT for token classification (POS)
pos_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(pos_label_list),
    id2label=pos_id2label,
    label2id=pos_label2id
)
print("\nPOS model loaded. Output layer size:", pos_model.classifier.out_features)

### 3.2 Chunking Model

In [ ]:
# Build label mappings for Chunking
chunk_id2label = {i: label for i, label in enumerate(chunk_label_list)}
chunk_label2id = {label: i for i, label in enumerate(chunk_label_list)}

print(f"Number of Chunk labels: {len(chunk_label_list)}")
print("Chunk id2label:", chunk_id2label)

# Load DistilBERT for token classification (Chunking)
chunk_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(chunk_label_list),
    id2label=chunk_id2label,
    label2id=chunk_label2id
)
print("\nChunk model loaded. Output layer size:", chunk_model.classifier.out_features)

## Task 4: Training

We use Hugging Face `Trainer` with:
- Learning rate: `2e-5` (standard for BERT fine-tuning)
- Epochs: `3`
- Batch size: `16`
- Weight decay regularization

In [ ]:
# Data collator pads sequences in a batch to the same length dynamically
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# Load seqeval for sequence-based evaluation
seqeval = evaluate.load("seqeval")

print("Data collator and seqeval metric loaded.")

In [ ]:
def build_compute_metrics(label_list):
    """
    Factory function: returns a compute_metrics function specific to
    the given label_list (POS or Chunk).
    Uses seqeval which expects IOB-format lists of label strings.
    """
    def compute_metrics(p):
        predictions, labels = p
        # Get the predicted class for each token (argmax over vocab)
        predictions = np.argmax(predictions, axis=2)

        # Convert integer IDs → label strings, skipping -100
        true_predictions = [
            [label_list[pred] for pred, lab in zip(prediction, label)
             if lab != -100]
            for prediction, label in zip(predictions, labels)
        ]
        true_labels = [
            [label_list[lab] for lab in label if lab != -100]
            for label in labels
        ]

        results = seqeval.compute(
            predictions=true_predictions,
            references=true_labels
        )
        return {
            "precision": results["overall_precision"],
            "recall":    results["overall_recall"],
            "f1":        results["overall_f1"],
            "accuracy":  results["overall_accuracy"]
        }
    return compute_metrics

print("Metrics function factory defined.")

### 4.1 Train POS Tagging Model

In [ ]:
pos_args = TrainingArguments(
    output_dir="./pos_model",
    evaluation_strategy="epoch",     # Evaluate at end of every epoch
    save_strategy="epoch",
    learning_rate=2e-5,              # Standard BERT fine-tuning LR
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,               # L2 regularization
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=200,
    report_to="none"                 # Disable wandb logging
)

pos_trainer = Trainer(
    model=pos_model,
    args=pos_args,
    train_dataset=tokenized_pos["train"],
    eval_dataset=tokenized_pos["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=build_compute_metrics(pos_label_list)
)

print("Starting POS model training...")
pos_trainer.train()
print("POS training complete!")

### 4.2 Train Chunking Model

In [ ]:
chunk_args = TrainingArguments(
    output_dir="./chunk_model",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=200,
    report_to="none"
)

chunk_trainer = Trainer(
    model=chunk_model,
    args=chunk_args,
    train_dataset=tokenized_chunk["train"],
    eval_dataset=tokenized_chunk["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=build_compute_metrics(chunk_label_list)
)

print("Starting Chunking model training...")
chunk_trainer.train()
print("Chunking training complete!")

## Task 5: Evaluation

We evaluate both models on the **test set** using `seqeval`, which computes:
- **Precision** – Of all predicted labels, how many are correct?
- **Recall** – Of all true labels, how many did we find?
- **F1 Score** – Harmonic mean of precision and recall

In [ ]:
# --- POS Evaluation on Test Set ---
print("=" * 50)
print("POS TAGGING EVALUATION (Test Set)")
print("=" * 50)
pos_results = pos_trainer.evaluate(tokenized_pos["test"])
for k, v in pos_results.items():
    print(f"  {k:30s}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

In [ ]:
# --- Chunking Evaluation on Test Set ---
print("=" * 50)
print("CHUNKING EVALUATION (Test Set)")
print("=" * 50)
chunk_results = chunk_trainer.evaluate(tokenized_chunk["test"])
for k, v in chunk_results.items():
    print(f"  {k:30s}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

In [ ]:
# Summary comparison table
print("\n" + "=" * 55)
print(f"{'Metric':<20} {'POS Tagging':>15} {'Chunking':>15}")
print("-" * 55)
for metric in ['eval_precision', 'eval_recall', 'eval_f1', 'eval_accuracy']:
    m_short = metric.replace('eval_', '').capitalize()
    p_val = pos_results.get(metric, 'N/A')
    c_val = chunk_results.get(metric, 'N/A')
    p_str = f"{p_val:.4f}" if isinstance(p_val, float) else str(p_val)
    c_str = f"{c_val:.4f}" if isinstance(c_val, float) else str(c_val)
    print(f"{m_short:<20} {p_str:>15} {c_str:>15}")
print("=" * 55)

## Task 6: Inference

### 6.1 Inference Helper Function

In [ ]:
def predict_tags(sentence, model, tokenizer, id2label):
    """
    Predict token-level tags for a raw input sentence.

    Steps:
    1. Split sentence into words
    2. Tokenize with BERT tokenizer (subwords)
    3. Run forward pass through model
    4. Decode predictions → label strings
    5. Assign only the first subword prediction to each word
    """
    words = sentence.split()

    # Tokenize
    inputs = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True
    )
    word_ids = inputs.word_ids()

    # Move to correct device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    model.to(device)

    # Forward pass (no gradient needed during inference)
    with torch.no_grad():
        outputs = model(**inputs)

    # Decode predictions
    predictions = torch.argmax(outputs.logits, dim=-1).squeeze().tolist()

    # Map: for each original word, take the prediction from its first subword
    word_predictions = {}
    for token_idx, word_id in enumerate(word_ids):
        if word_id is not None and word_id not in word_predictions:
            word_predictions[word_id] = id2label[predictions[token_idx]]

    return words, [word_predictions[i] for i in range(len(words))]

print("Inference function ready.")

### 6.2 Run Inference on Custom Sentences

In [ ]:
# Test sentences
test_sentences = [
    "John works at Google in California",
    "The quick brown fox jumps over the lazy dog",
    "She is studying machine learning at the university"
]

for sent in test_sentences:
    print("\n" + "=" * 60)
    print(f"Input: {sent}")
    print("-" * 60)

    # POS Prediction
    words, pos_preds = predict_tags(
        sent, pos_model, tokenizer, pos_id2label
    )
    print("POS Tags:")
    print(f"  {' '.join(f'{w}/{t}' for w, t in zip(words, pos_preds))}")

    # Chunk Prediction
    _, chunk_preds = predict_tags(
        sent, chunk_model, tokenizer, chunk_id2label
    )
    print("Chunk Tags:")
    print(f"  {' '.join(f'{w}/{t}' for w, t in zip(words, chunk_preds))}")

### 6.3 Formatted Output for Sample Sentence

In [ ]:
# Detailed table view for the main example
main_sent = "John works at Google in California"
words, pos_preds   = predict_tags(main_sent, pos_model, tokenizer, pos_id2label)
_, chunk_preds     = predict_tags(main_sent, chunk_model, tokenizer, chunk_id2label)

print(f"\nSentence: '{main_sent}'\n")
print(f"{'Word':<15} {'POS Tag':<12} {'Chunk Tag'}")
print("-" * 40)
for w, p, c in zip(words, pos_preds, chunk_preds):
    print(f"{w:<15} {p:<12} {c}")

## Task 7: Comparison – POS Tagging vs Chunking

### 7.1 Conceptual Comparison

In [ ]:
comparison = {
    "Aspect": [
        "Task Level",
        "Granularity",
        "Label Format",
        "Example Labels",
        "Difficulty",
        "Use Case",
        "Context Needed",
        "Model F1 (ours)"
    ],
    "POS Tagging": [
        "Grammar-level (word)",
        "Per-word syntactic role",
        "Single flat labels",
        "NN, VBZ, NNP, JJ, IN",
        "Easier – 45 labels",
        "Spell checking, parsing, IR",
        "Local context (1-2 tokens)",
        f"{pos_results.get('eval_f1', 'N/A'):.4f}" if isinstance(pos_results.get('eval_f1'), float) else "See eval"
    ],
    "Chunking": [
        "Phrase-level (multi-word)",
        "Multi-word phrase spans",
        "IOB2 (B-/I-/O)",
        "B-NP, I-NP, B-VP, O",
        "Medium – phrase spans",
        "Info extraction, Q&A, MT",
        "Phrase-span context",
        f"{chunk_results.get('eval_f1', 'N/A'):.4f}" if isinstance(chunk_results.get('eval_f1'), float) else "See eval"
    ]
}

print("\nPOS TAGGING vs CHUNKING – COMPARISON")
print("=" * 80)
print(f"{'Aspect':<22} {'POS Tagging':<30} {'Chunking'}")
print("-" * 80)
for a, p, c in zip(comparison["Aspect"], comparison["POS Tagging"], comparison["Chunking"]):
    print(f"{a:<22} {p:<30} {c}")
print("=" * 80)

In [ ]:
# Show on same sentence
example = "The big company hired skilled engineers"
words, pos = predict_tags(example, pos_model, tokenizer, pos_id2label)
_, chunks  = predict_tags(example, chunk_model, tokenizer, chunk_id2label)

print(f"\nSentence: '{example}'")
print("\nPOS Tagging (word-level syntactic roles):")
for w, p in zip(words, pos):
    print(f"  {w:12} → {p}")

print("\nChunking (phrase-span grouping):")
for w, c in zip(words, chunks):
    print(f"  {w:12} → {c}")

print("\nObservation:")
print("  POS assigns individual grammatical roles per word.")
print("  Chunking groups words into phrases (NP='noun phrase', VP='verb phrase', etc.)")
print("  B- marks phrase start, I- marks continuation, O marks outside any phrase.")

## Task 8: Report / Blog

---

### Differences Between POS Tagging and Chunking

**Part-of-Speech (POS) Tagging** assigns a grammatical role to each individual word in a sentence — for example, whether a word is a noun (`NN`), a verb (`VBZ`), an adjective (`JJ`), or a preposition (`IN`). It operates at the **word level** and captures local, syntactic identity.

**Chunking** (also called *shallow parsing*) groups consecutive words into meaningful phrases such as noun phrases (`NP`) or verb phrases (`VP`). It uses the **IOB2 format**: `B-` marks the beginning of a phrase, `I-` marks continuation, and `O` means the word is outside any phrase. Chunking captures **multi-word syntactic structure**, sitting between word-level POS and full parse trees.

In practical terms:
- POS: `"The/DT big/JJ company/NN hired/VBD skilled/JJ engineers/NNS"`
- Chunking: `"The big company" → [B-NP I-NP I-NP]`, `"hired" → [B-VP]`, `"skilled engineers" → [B-NP I-NP]`

---

### Challenges Faced

1. **Subword Tokenization Alignment**: DistilBERT's WordPiece tokenizer splits words like *"playing"* into `["play", "##ing"]`. Each subword gets its own embedding, but we only want **one label per original word**. The solution was to assign the real label to the **first subword** and `-100` to all others — `-100` is ignored by PyTorch's loss function.

2. **Special Token Handling**: `[CLS]` (sentence start) and `[SEP]` (sentence end) tokens have `word_id = None`. These also receive `-100` so they don't affect training.

3. **Label Count Asymmetry**: CoNLL-2003 has **45 POS tags** vs only **23 chunk tags**. Training two separate models ensured the output head was correctly sized for each task.

4. **seqeval Requirements**: The `seqeval` library expects predictions as **lists of string labels**, not integer IDs. It also expects **IOB-compatible format**, which chunk labels already satisfy. POS tags technically don't use IOB, but seqeval still computes overall metrics correctly.

---

### Observations and Insights

- **BERT's bidirectional attention** is highly effective for both tasks — it captures context from both directions, helping resolve ambiguous words (e.g., *"bank"* as noun vs. verb).
- **POS tagging** converges faster and achieves higher F1 because each token's label depends primarily on local context — and DistilBERT's attention captures that well even in early epochs.
- **Chunking** requires understanding phrase-span boundaries, which involves integrating context across multiple tokens — slightly harder, but BERT's self-attention handles this naturally.
- The **DataCollatorForTokenClassification** is crucial — it pads `input_ids` with the tokenizer's pad token, and pads `labels` with `-100`, ensuring padding positions are always ignored in the loss.
- Fine-tuning took only **3 epochs** with a learning rate of `2e-5` to achieve strong performance — demonstrating the power of transfer learning from pre-trained BERT weights.

---

### Key Takeaways

| Insight | Detail |
|---------|--------|
| BERT transfers well | Pre-trained weights give a massive head start on NLP tasks |
| Label alignment is critical | Incorrect alignment silently corrupts training |
| -100 is essential | Without it, special/subword tokens add noise to the loss |
| seqeval > accuracy | Accuracy overcounts padding; seqeval evaluates meaningful tokens only |
| Two tasks, one architecture | The same DistilBERT backbone handles both POS and chunking — only the head changes |

## Save Models

In [ ]:
# Save both fine-tuned models and tokenizer
pos_model.save_pretrained("./saved_pos_model")
chunk_model.save_pretrained("./saved_chunk_model")
tokenizer.save_pretrained("./saved_tokenizer")
print("Models and tokenizer saved successfully.")

## Load Saved Model and Run Inference

In [ ]:
# Demonstrate loading a saved model
loaded_pos_model = AutoModelForTokenClassification.from_pretrained("./saved_pos_model")
loaded_tokenizer = AutoTokenizer.from_pretrained("./saved_tokenizer")
loaded_id2label  = loaded_pos_model.config.id2label

sentence = "Barack Obama was the president of the United States"
words, preds = predict_tags(sentence, loaded_pos_model, loaded_tokenizer, loaded_id2label)

print(f"\nInference with loaded POS model:")
print(f"Input: {sentence}")
print("\n{:<15} {}".format("Word", "POS Tag"))
print("-" * 30)
for w, p in zip(words, preds):
    print(f"{w:<15} {p}")

---
## Summary

| Task | Status |
|------|--------|
| Dataset Selection (CoNLL-2003) | ✅ Complete |
| Data Preprocessing + Label Alignment | ✅ Complete |
| DistilBERT Model Setup (POS + Chunk) | ✅ Complete |
| Training with Hugging Face Trainer | ✅ Complete |
| Evaluation using seqeval (P/R/F1) | ✅ Complete |
| Inference on Custom Sentences | ✅ Complete |
| Comparison of POS vs Chunking | ✅ Complete |
| Report / Blog | ✅ Complete |

---
*Submitted by Om | Innomatics Research Labs – Agentic AI Intern*